# FAST-NUCES Karachi — Fall 2026
## AI4015: Agentic AI — Assignment Beta
### Question 4: Sentiment Classification Evaluation & Justification Metrics
**Student Name:** Muhammad John Raza  
**Roll Number:** `23K-0069`  
**Derived Parameters:** $R = 69$, $D = 0$  
---
### Overview & Objectives
This notebook implements comprehensive evaluation across two prompting regimes for LLM sentiment classification on 300 customer reviews:
1. **Stage 1 (Zero-Shot Baseline):** Evaluates overall accuracy with 95% Wilson and Wald confidence intervals, class-wise Precision/Recall/F1, and Macro/Weighted/Micro-averaged F1.
2. **Error Diagnosis & Prompt Engineering:** Diagnoses systematic misclassifications of neutral reviews and specifies prompt interventions.
3. **Stage 2 (Improved Few-Shot + Neutral Definition):** Re-evaluates all metrics and quantifies comparative deltas.
4. **Prompt Stability Analysis:** Calculates sample mean, standard deviation, and coefficient of variation across 5 shuffled few-shot runs.
5. **Justification Quality Evaluation:** Evaluates candidate vs. gold justification using BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L, and METEOR.
6. **Perplexity Analysis:** Derives exact token-level surprisal and sequence perplexity from token generation probabilities.

In [ ]:
import math
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

# Initialize tokenizer resources
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

print('Environment initialized successfully.')

---
### 1. Confusion Matrices Definition
Rows represent Actual sentiment classes `[Positive, Negative, Neutral]` ($N = 300$).  
Columns represent Predicted sentiment classes `[Positive, Negative, Neutral]`.

In [ ]:
classes = ['Positive', 'Negative', 'Neutral']

# Stage 1: Zero-shot baseline
cm_stage1 = [
    [100, 5, 15],  # Actual Pos: 120
    [8, 80, 12],   # Actual Neg: 100
    [20, 10, 50]   # Actual Neu: 80
]

# Stage 2: Improved few-shot prompt
cm_stage2 = [
    [106, 4, 10],  # Actual Pos: 120
    [6, 84, 10],   # Actual Neg: 100
    [9, 7, 64]     # Actual Neu: 80
]

df_s1 = pd.DataFrame(cm_stage1, index=[f'Actual {c}' for c in classes], columns=[f'Pred {c}' for c in classes])
df_s1['Row Total'] = df_s1.sum(axis=1)
print('--- Stage 1 Confusion Matrix ---')
print(df_s1)

df_s2 = pd.DataFrame(cm_stage2, index=[f'Actual {c}' for c in classes], columns=[f'Pred {c}' for c in classes])
df_s2['Row Total'] = df_s2.sum(axis=1)
print('\n--- Stage 2 Confusion Matrix ---')
print(df_s2)

---
### 2. Comprehensive Metric Evaluation Function
Calculates:
- Overall Accuracy: $\text{Acc} = \frac{\sum \text{diag}}{N}$
- 95% Confidence Intervals:
  - **Wald (Normal) Interval:** $\hat{p} \pm z \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$
  - **Wilson Score Interval:** $\frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$
- Per-class Precision, Recall, and $F_1$
- Macro-averaged $F_1$, Weighted-averaged $F_1$, and Micro-averaged $F_1$.

In [ ]:
def evaluate_confusion_matrix(cm, label=''):
    n = sum(sum(r) for r in cm)
    correct = sum(cm[i][i] for i in range(len(cm)))
    acc = correct / n
    z = 1.959964  # 95% two-sided critical value
    
    # Wald Interval
    se_wald = math.sqrt((acc * (1.0 - acc)) / n)
    wald_ci = (acc - z * se_wald, acc + z * se_wald)
    
    # Wilson Score Interval
    denom = 1.0 + (z**2) / n
    center = (acc + (z**2) / (2.0 * n)) / denom
    margin = (z * math.sqrt((acc * (1.0 - acc) / n) + ((z**2) / (4.0 * (n**2))))) / denom
    wilson_ci = (center - margin, center + margin)
    
    row_totals = [sum(row) for row in cm]
    col_totals = [sum(cm[i][j] for i in range(len(cm))) for j in range(len(cm))]
    
    records = []
    f1_scores = []
    for i, c in enumerate(classes):
        tp = cm[i][i]
        fp = col_totals[i] - tp
        fn = row_totals[i] - tp
        tn = n - tp - fp - fn
        prec = tp / col_totals[i] if col_totals[i] > 0 else 0.0
        rec = tp / row_totals[i] if row_totals[i] > 0 else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        f1_scores.append(f1)
        records.append({
            'Class': c,
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Precision': prec, 'Recall': rec, 'F1': f1,
            'Support': row_totals[i]
        })
        
    macro_f1 = sum(f1_scores) / len(f1_scores)
    weighted_f1 = sum(f * s for f, s in zip(f1_scores, row_totals)) / n
    micro_f1 = acc
    
    return {
        'label': label,
        'n': n, 'correct': correct, 'accuracy': acc,
        'wald_ci': wald_ci, 'wilson_ci': wilson_ci,
        'df_classes': pd.DataFrame(records),
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'micro_f1': micro_f1
    }

res_s1 = evaluate_confusion_matrix(cm_stage1, 'Stage 1 (Zero-Shot)')
res_s2 = evaluate_confusion_matrix(cm_stage2, 'Stage 2 (Few-Shot)')

print(f"=== {res_s1['label']} ===")
print(f"Accuracy: {res_s1['accuracy']:.4f} ({res_s1['correct']}/{res_s1['n']})")
print(f"95% Wald CI:   [{res_s1['wald_ci'][0]:.4f}, {res_s1['wald_ci'][1]:.4f}]")
print(f"95% Wilson CI: [{res_s1['wilson_ci'][0]:.4f}, {res_s1['wilson_ci'][1]:.4f}]")
print(res_s1['df_classes'][['Class', 'Precision', 'Recall', 'F1', 'Support']].to_string(index=False))
print(f"Macro-F1: {res_s1['macro_f1']:.4f} | Weighted-F1: {res_s1['weighted_f1']:.4f} | Micro-F1: {res_s1['micro_f1']:.4f}")

---
### 3. Stage 1 Error Interpretation & Prompt Interventions
#### Failure Mode Analysis:
1. **Severe Neutral Class Degradation:** The baseline zero-shot model achieves only $62.50\%$ recall and $64.94\%$ precision on Neutral reviews ($F_1 = 0.6369$).
2. **Polarization of Mixed Reviews:** Out of 80 true Neutral reviews, $20$ ($25.0\%$) were erroneously classified as Positive, and $10$ ($12.5\%$) as Negative. Without a formal semantic definition, the LLM over-indexes on isolated emotional adjectives (e.g. "good" or "late") and forces the review into a binary positive/negative classification.
3. **Reciprocal Leakage:** $15$ Positive and $12$ Negative reviews were classified as Neutral, demonstrating boundary confusion.

#### Prescribed Interventions (Implemented in Stage 2):
- Provide an **explicit semantic boundary definition** for Neutral: *"factual, mixed, or no clear opinion"*.
- Introduce **6 few-shot exemplars** (2 per class), deliberately showcasing borderline mixed reviews (e.g., product good vs delivery slow).

---
### 4. Stage 2 Evaluation & Direct Comparison
Now we evaluate Stage 2 and tabulate the performance deltas.

In [ ]:
print(f"=== {res_s2['label']} ===")
print(f"Accuracy: {res_s2['accuracy']:.4f} ({res_s2['correct']}/{res_s2['n']})")
print(f"95% Wald CI:   [{res_s2['wald_ci'][0]:.4f}, {res_s2['wald_ci'][1]:.4f}]")
print(f"95% Wilson CI: [{res_s2['wilson_ci'][0]:.4f}, {res_s2['wilson_ci'][1]:.4f}]")
print(res_s2['df_classes'][['Class', 'Precision', 'Recall', 'F1', 'Support']].to_string(index=False))
print(f"Macro-F1: {res_s2['macro_f1']:.4f} | Weighted-F1: {res_s2['weighted_f1']:.4f} | Micro-F1: {res_s2['micro_f1']:.4f}")

# Direct Comparative Table
comp_data = [
    ('Accuracy', res_s1['accuracy'], res_s2['accuracy'], res_s2['accuracy'] - res_s1['accuracy']),
    ('Macro-F1', res_s1['macro_f1'], res_s2['macro_f1'], res_s2['macro_f1'] - res_s1['macro_f1']),
    ('Weighted-F1', res_s1['weighted_f1'], res_s2['weighted_f1'], res_s2['weighted_f1'] - res_s1['weighted_f1']),
    ('Positive F1', res_s1['df_classes'].loc[0, 'F1'], res_s2['df_classes'].loc[0, 'F1'], res_s2['df_classes'].loc[0, 'F1'] - res_s1['df_classes'].loc[0, 'F1']),
    ('Negative F1', res_s1['df_classes'].loc[1, 'F1'], res_s2['df_classes'].loc[1, 'F1'], res_s2['df_classes'].loc[1, 'F1'] - res_s1['df_classes'].loc[1, 'F1']),
    ('Neutral F1', res_s1['df_classes'].loc[2, 'F1'], res_s2['df_classes'].loc[2, 'F1'], res_s2['df_classes'].loc[2, 'F1'] - res_s1['df_classes'].loc[2, 'F1']),
]
df_comp = pd.DataFrame(comp_data, columns=['Metric', 'Stage 1 (Zero-Shot)', 'Stage 2 (Few-Shot)', 'Delta (+/-)'])
print('\n=== Side-by-Side Comparison ===')
print(df_comp.to_string(index=False))

---
### 5. Prompt Stability Across 5 Few-Shot Shuffles
The 6 few-shot examples were permuted in 5 different orderings, yielding accuracies:
$$[0.847, 0.840, 0.853, 0.837, 0.850]$$

In [ ]:
runs = [0.847, 0.840, 0.853, 0.837, 0.850]
mean_val = statistics.mean(runs)
std_val = statistics.stdev(runs)
cv_val = (std_val / mean_val) * 100.0

print(f"Accuracies: {runs}")
print(f"Mean Accuracy:           {mean_val:.4f} ({mean_val*100:.2f}%)")
print(f"Sample Std Dev (s):      {std_val:.4f} ({std_val*100:.2f}%)")
print(f"Mean +/- Std Dev:        {mean_val:.4f} +/- {std_val:.4f} [{mean_val-std_val:.4f}, {mean_val+std_val:.4f}]")
print(f"Coefficient of Variation: {cv_val:.2f}%")

#### Stability Commentary:
- The standard deviation of $s = 0.0067$ ($0.67$ percentage points) and coefficient of variation of $0.80\%$ demonstrate **exceptional prompt stability**.
- Unlike fragile prompts where exemplar ordering triggers recency bias or class-priors shifting, the balanced 2-per-class exemplar structure combined with an explicit semantic definition provides a robust decision boundary across all permutations.

---
### 6. Justification Evaluation (NLP Metrics)
Reference and candidate justification texts for a mixed review (gold label: Neutral):
- **Reference:** `the delivery was late but the product quality is good`
- **Candidate:** `the product quality is good but delivery was slow`

Evaluating with BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L, and METEOR.

In [ ]:
ref_str = 'the delivery was late but the product quality is good'
cand_str = 'the product quality is good but delivery was slow'

ref_toks = ref_str.split()
cand_toks = cand_str.split()

# BLEU-4 calculation
def get_ngrams(seq, n):
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

print('=== N-Gram Precision Breakdown ===')
for n in range(1, 5):
    c_ng = get_ngrams(cand_toks, n)
    r_ng = list(get_ngrams(ref_toks, n))
    match = 0
    for g in c_ng:
        if g in r_ng:
            match += 1
            r_ng.remove(g)
    print(f"P_{n} ({n}-gram precision): {match}/{len(c_ng)} = {match/len(c_ng):.4f}")

bp = 1.0 if len(cand_toks) >= len(ref_toks) else math.exp(1.0 - len(ref_toks) / len(cand_toks))
smoothie = SmoothingFunction().method1
bleu4 = sentence_bleu([ref_toks], cand_toks, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
r_scores = scorer.score(ref_str, cand_str)

# METEOR
meteor_val = meteor_score([ref_toks], cand_toks)

print(f"\nBrevity Penalty: {bp:.4f}")
print(f"Sentence BLEU-4: {bleu4:.4f}")
print(f"ROUGE-1 F1:     {r_scores['rouge1'].fmeasure:.4f} (P={r_scores['rouge1'].precision:.4f}, R={r_scores['rouge1'].recall:.4f})")
print(f"ROUGE-2 F1:     {r_scores['rouge2'].fmeasure:.4f} (P={r_scores['rouge2'].precision:.4f}, R={r_scores['rouge2'].recall:.4f})")
print(f"ROUGE-L F1:     {r_scores['rougeL'].fmeasure:.4f} (P={r_scores['rougeL'].precision:.4f}, R={r_scores['rougeL'].recall:.4f})")
print(f"METEOR Score:   {meteor_val:.4f}")

---
### 7. Perplexity Computation & Surprisal Analysis
Given token generation probabilities for the 9 candidate tokens:
$$[0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20]$$

Perplexity is defined as the exponential of the average negative log-likelihood (NLL):
$$\text{PPL} = \exp \left( -\frac{1}{N} \sum_{i=1}^N \ln P(w_i) \right)$$

In [ ]:
token_probs = [0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20]
n_tokens = len(token_probs)

log_p = [math.log(p) for p in token_probs]
sum_neg_log = sum(-lp for lp in log_p)
avg_nll = sum_neg_log / n_tokens
ppl = math.exp(avg_nll)

df_ppl = pd.DataFrame({
    'Token': cand_toks,
    'Prob P(w)': token_probs,
    'ln P(w)': log_p,
    'Surprisal (-ln P)': [-lp for lp in log_p]
})
print('=== Token-Level Surprisal Analysis ===')
print(df_ppl.to_string(index=False))
print('-'*50)
print(f"Sum of Surprisals (Total NLL): {sum_neg_log:.4f} nats")
print(f"Average NLL per Token:          {avg_nll:.4f} nats")
print(f"Sequence Perplexity (PPL):     {ppl:.4f}")

#### Perplexity Interpretation:
1. **Branching Factor:** A perplexity of $\mathbf{2.32}$ indicates that the language model is, on average, choosing among approximately $2.3$ equally likely candidate tokens at each generation step. This represents very high fluency and confidence.
2. **Information Density & Surprisal Dynamics:**
   - **High Certainty:** Syntactic scaffolding tokens such as `is` ($0.80$) and sentiment predicates such as `good` ($0.70$) have very low surprisal ($0.22$ and $0.36$ nats).
   - **High Surprisal:** The discourse contrastive connective `but` ($P = 0.30$, surprisal $1.20$), topic shift `delivery` ($P = 0.25$, surprisal $1.39$), and lexical substitution `slow` ($P = 0.20$, surprisal $1.61$) carry the highest surprisals. This aligns with linguistic intuition: lexical choices and transitions encode the bulk of the sentence's semantic payload.